# 03 - Throughput Analysis

Analyze throughput over time, identify instabilities, and measure sustained performance.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

try:
    %store -r df
    print(f"Loaded {len(df):,} events")
except:
    np.random.seed(42)
    n = 50000
    df = pd.DataFrame({
        "timestamp_utc_iso": pd.date_range("2025-01-01", periods=n, freq="10ms"),
        "latency_us": np.random.lognormal(6, 0.5, n).astype(int),
        "worker_id": np.random.randint(0, 4, n),
    })

if "timestamp" not in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp_utc_iso"])


In [ ]:
# Calculate throughput
df["second"] = df["timestamp"].dt.floor("S")
throughput = df.groupby("second").size()

print(f"Duration: {len(throughput)} seconds")
print(f"Total: {len(df):,} messages")
print(f"Mean: {throughput.mean():.1f} msg/s")
print(f"Max: {throughput.max()} msg/s")
print(f"CV: {throughput.std() / throughput.mean():.2%}")


In [ ]:
# Throughput over time
fig, ax = plt.subplots(figsize=(14, 5))
time_seconds = (throughput.index - throughput.index.min()).total_seconds()
ax.plot(time_seconds, throughput.values, linewidth=1, alpha=0.7)
ax.fill_between(time_seconds, throughput.values, alpha=0.3)
ax.axhline(throughput.mean(), color="red", linestyle="--", label=f"Mean: {throughput.mean():.0f}")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Messages/second")
ax.set_title("Throughput Over Time")
ax.legend()
plt.tight_layout()
plt.show()


# 03 - Throughput Analysis

Analyze throughput over time and identify performance instabilities.

## Objectives
- Compute throughput time series
- Identify throughput degradations
- Analyze per-worker throughput
- Detect instabilities and anomalies


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXPERIMENT_ID = "exp_2025_0101_001"  # Change this
DATA_PATH = f"../data/{EXPERIMENT_ID}/merged/merged.parquet"

df = pd.read_parquet(DATA_PATH)
if "timestamp_utc_iso" in df.columns:
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc_iso"])
print(f"Loaded {len(df):,} records")


In [ ]:
# Compute throughput over time
df_sorted = df.sort_values("timestamp_utc").set_index("timestamp_utc")
throughput = df_sorted.resample("1S").size()
time_seconds = (throughput.index - throughput.index[0]).total_seconds()

# Summary stats
duration = time_seconds.max()
print(f"Duration: {duration:.2f} seconds")
print(f"Total operations: {len(df):,}")
print(f"Overall throughput: {len(df) / duration:.2f} ops/s")
print(f"Rolling mean: {throughput.mean():.2f} ops/s")
print(f"Rolling std: {throughput.std():.2f} ops/s")


In [ ]:
# Throughput plot
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(time_seconds, throughput.values, alpha=0.5, color="#2196F3", label="Instantaneous")
rolling_5s = throughput.rolling(window=5, min_periods=1).mean()
ax.plot(time_seconds, rolling_5s.values, color="#E53935", linewidth=2, label="5s Rolling Avg")
ax.axhline(throughput.mean(), color="#4CAF50", linestyle="--", label=f"Mean: {throughput.mean():.1f}")

ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Throughput (ops/second)")
ax.set_title(f"Throughput Over Time - {EXPERIMENT_ID}")
ax.legend()
plt.tight_layout()
plt.show()
